## Regularized Linear Regression (Elastic Net)

In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV

In [43]:
df = pd.read_csv("../dataset/train.csv")
df = df.drop(columns=['id'])

In [44]:
# Step 1
y = df['exam_score']
X = df.drop(columns=['exam_score'])

In [45]:
# Step 2
# Assuming students are independent, therefore shuffle=True
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [46]:
# Step 3
# Linear models require different preprocessing for numeric vs categorical data
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

In [47]:
# Step 4
# One-Hot encoding of categorical features
# It Works perfectly with Elastic Net, Regularization will: Shrink unimportant categories (e.g., gender)
categorical_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [48]:
# Step 5
"""
standardization (z-score scaling) to all numerical features:
Elastic Net is scale-sensitive
L1 and L2 penalties depend on coefficient magnitude
Without scaling, features with large units (e.g., attendance %) dominate
Scaling ensures each numeric feature is penalized equally
Allows Elastic Net to:
    shrink weak predictors
    retain strong ones (study_hours)
"""
numerical_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [49]:
# Step 6
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [50]:
# Step 7
"""
Evaluation metric: RMSE (computed on validation set)
Mixing parameter α (a.k.a. l1_ratio) strictly between 0 and 1

"""
model = ElasticNet(
    alpha=0.1,          # moderate regularization
    l1_ratio=0.5,       # balanced L1/L2
    max_iter=10000,
    random_state=42
)

In [51]:
# Step 8
# Full-Pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [52]:
# Step 9
# Fitting the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'study_hours', 'class_attendance', 'sleep_hours'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['gender', 'course', 'internet_access', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty'],
      dtype='object'))])),
                ('model',
                 ElasticNet(alpha=0.1, max_iter=10000, random_state=42))])

In [55]:
y_pred = pipeline.predict(X_valid)
rmse = root_mean_squared_error(y_valid, y_pred)

print(f"Baseline Elastic Net RMSE: {rmse:.4f}")

Baseline Elastic Net RMSE: 9.1749


In [57]:
# Step 10
# HyperParameter Tuning
param_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10],
    "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

In [58]:
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.9}
Best CV RMSE: 8.896864418502343


In [ ]:
# Validating the 'best model' after GridSearch
best_model = grid.best_estimator_

y_pred = best_model.predict(X_valid) 
rmse = root_mean_squared_error(y_valid, y_pred) 
print("Validation RMSE:", rmse)

Validation RMSE: 8.88649659775039


In [60]:
# Step 11
# Retraining with best parameters on the whole training set.
best_params = grid.best_params_

final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", ElasticNet(
            alpha=best_params["model__alpha"],
            l1_ratio=best_params["model__l1_ratio"],
            max_iter=10000,
            random_state=42
        ))
    ]
)

In [61]:
# Step 11
final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'study_hours', 'class_attendance', 'sleep_hours'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['gender', 'course', 'internet_access', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty'],
      dtype='object'))])),
                ('model',
                 ElasticNet(alpha=0.001, l1_ratio=0.9, max_iter=10000,
                            random_state=42))])

In [66]:
# Step 12
# Iporting the test set
test_df = pd.read_csv("../dataset/test.csv")
ids = test_df['id']
test_df = test_df.drop(columns=['id'])

In [68]:
# Step 13
# Testing on the test set
test_pred = final_model.predict(test_df)

In [71]:
# Step 14
# Submission
submission = pd.DataFrame({
    'id': ids,
    'exam_score': test_pred
})
submission.to_csv('Elastic_Net_Regression.csv', index=False)

In [67]:
test_df.head(2)

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy


In [70]:
df.head(3)

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
